In [1]:
import csv

condition_mapping_file = "../../resources/CCSCM.csv"
procedure_mapping_file = "../../resources/CCSPROC.csv"
drug_file = "../../resources/ATC.csv"

condition_dict = {}
with open(condition_mapping_file, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        condition_dict[row['code']] = row['name'].lower()

procedure_dict = {}
with open(procedure_mapping_file, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        procedure_dict[row['code']] = row['name'].lower()

drug_dict = {}
with open(drug_file, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        if row['level'] == '3.0':
            drug_dict[row['code']] = row['name'].lower()


In [2]:
from tqdm import tqdm
import json
import os

cond_ent = set()
cond_rel = set()

file_dir = "../../graphs/condition/CCSCM"

# Create directory if it doesn't exist
os.makedirs(file_dir, exist_ok=True)

for key in condition_dict.keys():
    file = f"{file_dir}/{key}.txt"
    with open(file=file, mode='r', encoding='utf-8') as f:
        lines = f.readlines()
    
    for line in lines:
        parsed = line.split('\t')
        if len(parsed) == 3:
            h, r, t = line.split('\t')
            t = t[:-1]
            cond_ent.add(h)
            cond_ent.add(t)
            cond_rel.add(r)


cond_id2ent = {index: value for index, value in enumerate(cond_ent)}
cond_ent2id = {value: index for index, value in enumerate(cond_ent)}
cond_id2rel = {index: value for index, value in enumerate(cond_rel)}
cond_rel2id = {value: index for index, value in enumerate(cond_rel)}

out_file_id2ent = f"{file_dir}/id2ent.json"
out_file_ent2id = f"{file_dir}/ent2id.json"
out_file_id2rel = f"{file_dir}/id2rel.json"
out_file_rel2id = f"{file_dir}/rel2id.json"

with open(out_file_id2ent, 'w', encoding='utf-8') as file:
    json.dump(cond_id2ent, file, indent=6)
with open(out_file_ent2id, 'w', encoding='utf-8') as file:
    json.dump(cond_ent2id, file, indent=6)
with open(out_file_id2rel, 'w', encoding='utf-8') as file:
    json.dump(cond_id2rel, file, indent=6)
with open(out_file_rel2id, 'w', encoding='utf-8') as file:
    json.dump(cond_rel2id, file, indent=6)


In [3]:
import json

file_dir = "../../graphs/condition/CCSCM"

file_id2ent = f"{file_dir}/id2ent.json"
file_ent2id = f"{file_dir}/ent2id.json"
file_id2rel = f"{file_dir}/id2rel.json"
file_rel2id = f"{file_dir}/rel2id.json"

with open(file_id2ent, 'r', encoding='utf-8') as file:
    cond_id2ent = json.load(file)
with open(file_ent2id, 'r', encoding='utf-8') as file:
    cond_ent2id = json.load(file)
with open(file_id2rel, 'r', encoding='utf-8') as file:
    cond_id2rel = json.load(file)
with open(file_rel2id, 'r', encoding='utf-8') as file:
    cond_rel2id = json.load(file)


In [4]:
from get_emb import embedding_retriever
import numpy as np
from tqdm import tqdm
import pickle
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

## get embedding for condition entities
def get_entity_embedding(idx, cond_id2ent):
    ent = cond_id2ent[str(idx)]
    embedding = embedding_retriever(term=ent)
    return idx, np.array(embedding)

cond_ent_emb = [None] * len(cond_id2ent)

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(get_entity_embedding, idx, cond_id2ent): idx 
               for idx in range(len(cond_id2ent))}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Entities"):
        idx, embedding = future.result()
        cond_ent_emb[idx] = embedding

stacked_embedding = np.vstack(cond_ent_emb)

# Create directory if it doesn't exist
os.makedirs(file_dir, exist_ok=True)

emb_pkl = f"{file_dir}/entity_embedding.pkl"

with open(emb_pkl, "wb") as file:
    pickle.dump(stacked_embedding, file)


Entities: 100%|██████████| 17321/17321 [56:24<00:00,  5.12it/s] 


In [5]:
from get_emb import embedding_retriever
import numpy as np
from tqdm import tqdm
import pickle
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

## get embedding for condition relations
def get_relation_embedding(idx, cond_id2rel):
    rel = cond_id2rel[str(idx)]
    embedding = embedding_retriever(term=rel)
    return idx, np.array(embedding)

cond_rel_emb = [None] * len(cond_id2rel)

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(get_relation_embedding, idx, cond_id2rel): idx 
               for idx in range(len(cond_id2rel))}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Relations"):
        idx, embedding = future.result()
        cond_rel_emb[idx] = embedding

stacked_embedding = np.vstack(cond_rel_emb)

# Create directory if it doesn't exist
os.makedirs(file_dir, exist_ok=True)

emb_pkl = f"{file_dir}/relation_embedding.pkl"

with open(emb_pkl, "wb") as file:
    pickle.dump(stacked_embedding, file)


Relations:   0%|          | 1/2338 [00:02<1:49:20,  2.81s/it]

Relations: 100%|██████████| 2338/2338 [03:32<00:00, 11.00it/s]
